# OpenMLS v7 AddCommit analysis

This notebook is the R front end for `statistics_analysis_openmls_v7.R`. Version 7 is intentionally AddCommit-only: it reads OpenMLS `events.csv` files, keeps create-side AddCommit profiling rows, writes cleaned plotting data and reports, and renders only the AddCommit heatmaps and suboperation LOESS/IQR plots.

## AddCommit scaling variables

- `N`: before-commit group size from the AddCommit create span's `member_count`.
- `k`: number of newly added members, preferring `added_members_count` and falling back only to observed equivalent AddCommit recipient/count fields.
- `C`: UpdatePath HPKE ciphertext count, preferring `sum_copath_resolution_sizes`.
- `F`: `filtered_direct_path_len`.
- `tree_artifact_bytes`: observed `ratchet_tree_bytes`; no tree-size proxy is used.
- `group_info_bytes`: observed serialized plaintext GroupInfo bytes; no encrypted-size proxy is used.

The backend reports missing spans or metrics explicitly. Historical CSVs generated before the AddCommit GroupInfo AEAD profiling span will not produce the GroupInfo AEAD plots.

In [ ]:
options(width = 120)

Sys.setenv(
  OPENMLS_V7_FILE_BATCH_SIZE = "1",
  OPENMLS_V7_CHUNK_ROWS = "200000",
  OPENMLS_V7_USE_CACHE = "true"
)

script_candidates <- c(
  "statistics_analysis_openmls_v7.R",
  file.path("statistics", "statistics_analysis_openmls_v7.R")
)
script_path <- script_candidates[file.exists(script_candidates)][1]
stopifnot(!is.na(script_path))
source(script_path)

out_dir <- openmls_v7_output_default
table_dir <- file.path(out_dir, "tables")
plot_dir <- file.path(out_dir, "plots")

force_backend <- FALSE
required_backend_files <- c(
  file.path(table_dir, "addcommit_metric_coverage.csv"),
  file.path(table_dir, "addcommit_span_mapping.csv"),
  file.path(table_dir, "plots_created.csv"),
  file.path(data_dir <- file.path(out_dir, "data"), "addcommit_plotting_rows.csv"),
  file.path(plot_dir, "addcommit_total_cpu_thread_time_thin_plate_heatmap.png")
)

if (force_backend || !all(file.exists(required_backend_files))) {
  result <- run_openmls_v7_analysis(render_plots = TRUE)
} else {
  message("Using existing backend outputs in ", out_dir, ". Set force_backend <- TRUE to regenerate.")
}


In [ ]:
# Export every created plot as an individual PDF and as one concatenated multipage PDF.
# This does not require rsvg/qpdf; each page is drawn directly from the ggplot object.
if (!exists("result") || is.null(result$plots$objects) || length(result$plots$objects) == 0) {
  result <- run_openmls_v7_analysis(render_plots = TRUE)
}

registry <- openmls_v7_plot_registry() |>
  dplyr::filter(filename %in% names(result$plots$objects)) |>
  dplyr::arrange(plot_kind, suboperation_key, metric_key)

pdf_page_dir <- file.path(out_dir, "plots_pdf_pages")
dir.create(pdf_page_dir, recursive = TRUE, showWarnings = FALSE)

pdf_pages <- character(nrow(registry))
for (i in seq_len(nrow(registry))) {
  filename <- registry$filename[[i]]
  plot_obj <- result$plots$objects[[filename]]
  pdf_page_path <- file.path(pdf_page_dir, sub("\\.png$", ".pdf", filename))

  grDevices::pdf(pdf_page_path, width = registry$width[[i]], height = registry$height[[i]], onefile = FALSE)
  print(plot_obj)
  grDevices::dev.off()
  pdf_pages[[i]] <- pdf_page_path
}

combined_pdf_path <- file.path(plot_dir, "addcommit_all_plots.pdf")
if (requireNamespace("qpdf", quietly = TRUE)) {
  qpdf::pdf_combine(input = pdf_pages, output = combined_pdf_path)
  combine_method <- "qpdf::pdf_combine"
} else {
  warning("Package qpdf is not installed; writing a direct multipage PDF fallback instead of concatenating individual PDFs.")
  grDevices::pdf(combined_pdf_path, width = 12, height = 6, onefile = TRUE)
  for (i in seq_len(nrow(registry))) {
    print(result$plots$objects[[registry$filename[[i]]]])
  }
  grDevices::dev.off()
  combine_method <- "grDevices::pdf fallback"
}

tibble::tibble(
  pdf_page_count = length(pdf_pages),
  individual_pdf_dir = pdf_page_dir,
  combine_method = combine_method,
  combined_pdf = combined_pdf_path
)


In [ ]:
coverage <- readr::read_csv(file.path(table_dir, "addcommit_metric_coverage.csv"), show_col_types = FALSE)
span_mapping <- readr::read_csv(file.path(table_dir, "addcommit_span_mapping.csv"), show_col_types = FALSE)
plots_created <- readr::read_csv(file.path(table_dir, "plots_created.csv"), show_col_types = FALSE)
plots_skipped <- readr::read_csv(file.path(table_dir, "plots_skipped.csv"), show_col_types = FALSE)

list(
  created_plots = nrow(plots_created),
  skipped_plots = nrow(plots_skipped),
  unavailable_inputs = coverage |> dplyr::filter(coverage_status != "available"),
  span_mapping = span_mapping |> dplyr::select(suboperation_key, raw_span_name, status, rows, note)
)
